# Feature Extraction: Transforming Text into Numbers with TF-IDF

## 1. Introduction
Now that our reviews are cleaned and normalized, the next challenge is to convert this text into a format that machine learning models can understand: **numbers**. 

We will use **TF-IDF (Term Frequency-Inverse Document Frequency)**. This technique doesn't just count words; it evaluates how important a word is to a document relative to the entire collection. 
- **TF (Term Frequency):** How often a word appears in a specific review.
- **IDF (Inverse Document Frequency):** Reduces the weight of words that appear very frequently across all reviews (like "product" or "bought") and increases the weight of words that are more unique and potentially more descriptive.

---


In [1]:
import os
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer

INPUT_PATH = "../data/preprocessed/cleaned_reviews.csv"
OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 2. Loading the Cleaned Data
We load the preprocessed dataset. During cleaning, some reviews (like those consisting only of stopwords) might have become empty strings. We'll ensure these are handled as empty strings rather than `NaN` to avoid errors during vectorization.


In [2]:
df = pd.read_csv(INPUT_PATH)

# Handle potential NaNs in cleaned text (e.g., if a review was only stopwords)
df['cleaned_review'] = df['cleaned_review'].fillna('')

X = df['cleaned_review']
y = df['label_binary']

print(f"Data loaded: {len(df)} samples.")


Data loaded: 40432 samples.


## 3. The TF-IDF Vectorizer
We configure our vectorizer with a richer set of parameters to maximize accuracy:
1.  **`ngram_range=(1, 3)`**: We capture unigrams, bigrams, and trigrams. Trigrams can capture more context, like "not very good".
2.  **`max_features=15000`**: We increase the vocabulary size to 15,000 terms to avoid discarding relevant rare terms.
3.  **`sublinear_tf=True`**: This applies sublinear TF scaling (1 + log(tf)), which dampens the effect of high-frequency words.
4.  **`min_df=5`**: Removes terms that appear in fewer than 5 documents, reducing noise.


In [3]:
vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 3),
    sublinear_tf=True,
    min_df=5
)

print("Fitting and transforming data...")
X_tfidf = vectorizer.fit_transform(X)

print(f"TF-IDF Matrix Shape: {X_tfidf.shape}")
print(f"Vocabulary Size: {len(vectorizer.get_feature_names_out())}")


Fitting and transforming data...
TF-IDF Matrix Shape: (40432, 15000)
Vocabulary Size: 15000


## 4. Persisting the Results
To ensure consistency across our five different models and the final GUI, we save the following artifacts:
- **`X_tfidf.pkl`**: The numerical features ready for training.
- **`y.pkl`**: The target labels.
- **`tfidf_vectorizer.pkl`**: The trained vectorizer. This is **CRITICAL** for the GUI, as it must use the exact same vocabulary and weights to process new user input.


In [4]:
joblib.dump(X_tfidf, os.path.join(OUTPUT_DIR, "X_tfidf.pkl"))
joblib.dump(y, os.path.join(OUTPUT_DIR, "y.pkl"))
joblib.dump(vectorizer, os.path.join(OUTPUT_DIR, "tfidf_vectorizer.pkl"))

print(f"All artifacts saved successfully to: {OUTPUT_DIR}")

All artifacts saved successfully to: ../data/processed
